# AlphaTrain: 5-Block Surrogate for Fast Self-Play

Train a small 5b×128ch model (2.9M params, 4x faster inference) for
self-play data generation. The big 10b model trains on the surrogate's games.

- V8 data (5.8M states from 2V ep6 teacher)
- Train from scratch (different architecture)
- 15 epochs, lr=1e-3

**Upload to Drive (`MyDrive/alphatrain/`):**
1. `colorlines_selfplay_train.tar.gz` — code
2. `selfplay_v8_policy.pt.gz` — V8 tensor

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, time

DRIVE = '/content/drive/MyDrive/alphatrain'

!cp {DRIVE}/colorlines_selfplay_train.tar.gz /content/
!cd /content && tar xzf colorlines_selfplay_train.tar.gz

# V8 data (no model needed — training from scratch)
os.makedirs('/content/alphatrain/data', exist_ok=True)
t0 = time.time()
!gzip -dc {DRIVE}/selfplay_v8_policy.pt.gz > /content/alphatrain/data/selfplay.pt
print(f'Data: {os.path.getsize("/content/alphatrain/data/selfplay.pt")/1e9:.1f} GB ({time.time()-t0:.0f}s)')

!pip install -q numpy numba scipy pytest

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {g.total_memory / 1e9:.1f} GB')

!cd /content && python -m pytest alphatrain/tests/test_model.py -v --tb=short 2>&1 | tail -5

In [ ]:
# Surrogate: 5b×128ch, from scratch, V8 data
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train \
    --tensor-file alphatrain/data/selfplay.pt \
    --amp --compile \
    --num-blocks 5 --channels 128 \
    --epochs 15 --batch-size 32768 --lr 1e-3 --warmup-epochs 3 \
    --copy-to /content/drive/MyDrive/alphatrain/surrogate_5b128_best.pt \
    --save-dir /content/checkpoints/surrogate

In [ ]:
# Copy checkpoints to Drive
import shutil, os, glob
DRIVE = '/content/drive/MyDrive/alphatrain'
for f in sorted(glob.glob('/content/checkpoints/surrogate/epoch_*.pt')):
    dst = f'{DRIVE}/surrogate_5b128_{os.path.basename(f)}'
    shutil.copy(f, dst)
    print(f'Saved {dst} ({os.path.getsize(dst)/1e6:.0f} MB)')
for f in ['best.pt', 'latest.pt']:
    src = f'/content/checkpoints/surrogate/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'{DRIVE}/surrogate_5b128_{f}')